In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Configuration
RAW_DATA_PATH = "../data/raw/net_zero_tracker.csv"
OUTPUT_DIR = "../data/interim"
TARGET_COL = 'Scope_3_coverage'
RANDOM_STATE = 42
TEST_SIZE = 0.2
VAL_SIZE = 0.2
MISSING_THRESHOLD = 0.5  # Drop columns with >50% missing

## 1. Load Raw Data

In [3]:
# Load raw data
df_raw = pd.read_csv(RAW_DATA_PATH, sep=';', quotechar='"', quoting=1, 
                     engine='python', on_bad_lines='skip', 
                     skipinitialspace=True, doublequote=True)

print(f"Raw data loaded: {df_raw.shape}")

Raw data loaded: (4183, 66)


In [4]:
# Filter to companies only and select relevant columns
keep_columns = [
    'ID_Code', 'Name', 'Country', 'Geographic_region', 'Entity_type',
    'Private_company', 'End_target', 'End_target_year', 'Status_of_end_target',
    'Interim_target', 'Interim_target_year', 'GHGs_covered',
    'Scope_1_coverage', 'Scope_2_coverage', 'Scope_3_coverage',
    'Published_plan', 'Reporting_mechanism', 'Accountability_delivery',
    'Carbon_credits', 'Separate_removal_target', 'Planning_removals',
    'Historical_emissions', 'Race_to_zero_member',
    'Company_annual_revenue', 'Industry', 'Employees', 'GHG_emissions'
]

# Filter to companies only
df = df_raw[df_raw['Entity_type'] == 'Company'][keep_columns].copy().reset_index(drop=True)
df = df.drop(columns=['Entity_type'])

print(f"Filtered to companies: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Filtered to companies: (2086, 26)
Columns: ['ID_Code', 'Name', 'Country', 'Geographic_region', 'Private_company', 'End_target', 'End_target_year', 'Status_of_end_target', 'Interim_target', 'Interim_target_year', 'GHGs_covered', 'Scope_1_coverage', 'Scope_2_coverage', 'Scope_3_coverage', 'Published_plan', 'Reporting_mechanism', 'Accountability_delivery', 'Carbon_credits', 'Separate_removal_target', 'Planning_removals', 'Historical_emissions', 'Race_to_zero_member', 'Company_annual_revenue', 'Industry', 'Employees', 'GHG_emissions']


## 2. Basic Cleaning (Before Split)

Only do cleaning that doesn't require statistics from the data:
- Drop columns with >50% missing
- Fix data types
- Remove rows with missing target

In [5]:
# Check missing values
missing_pct = df.isnull().sum() / len(df) * 100
print("Missing values per column:")
print(missing_pct.sort_values(ascending=False))

Missing values per column:
GHG_emissions              99.712368
Interim_target_year        38.734420
Status_of_end_target       21.188878
End_target_year            20.373921
Historical_emissions       14.141898
Separate_removal_target    12.655801
Planning_removals          11.169703
Carbon_credits             11.073826
Published_plan             10.498562
Accountability_delivery    10.258869
Reporting_mechanism         9.443912
Scope_3_coverage            9.252157
Scope_2_coverage            9.156280
Scope_1_coverage            9.108341
Employees                   7.526366
GHGs_covered                4.793864
Interim_target              3.163950
Industry                    0.431448
Company_annual_revenue      0.383509
End_target                  0.095877
Name                        0.000000
Private_company             0.000000
Geographic_region           0.000000
Country                     0.000000
Race_to_zero_member         0.000000
ID_Code                     0.000000
dtype: floa

In [6]:
# Drop columns with >50% missing
high_missing_cols = missing_pct[missing_pct > MISSING_THRESHOLD * 100].index.tolist()
print(f"\nDropping {len(high_missing_cols)} columns with >{MISSING_THRESHOLD*100:.0f}% missing:")
print(high_missing_cols)

df = df.drop(columns=high_missing_cols)
print(f"\nRemaining columns: {df.shape[1]}")


Dropping 1 columns with >50% missing:
['GHG_emissions']

Remaining columns: 25


In [7]:
# Fix numeric columns that may have string formatting (commas, etc.)
NUMERIC_CONVERT_COLS = ['GHG_emissions', 'Company_annual_revenue', 'Employees']

for col in NUMERIC_CONVERT_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', ''), errors='coerce')
        print(f"Converted {col} to numeric: {df[col].dtype}")

# Remove rows with missing target
rows_before = len(df)
df = df.dropna(subset=[TARGET_COL])
print(f"\nDropped {rows_before - len(df)} rows with missing target")

# Remove rows with missing Industry (required for group-based imputation)
rows_before = len(df)
df = df.dropna(subset=['Industry'])
print(f"Dropped {rows_before - len(df)} rows with missing Industry")

print(f"Final dataset size: {df.shape}")

Converted Company_annual_revenue to numeric: float64
Converted Employees to numeric: float64

Dropped 193 rows with missing target
Dropped 8 rows with missing Industry
Final dataset size: (1885, 25)


## 3. Train/Val/Test Split (BEFORE Imputation)

Critical: Split now so imputation statistics come only from training data.

In [8]:
# Separate features and target
IDENTIFIER_COLS = ['ID_Code', 'Name']
X = df.drop(columns=[TARGET_COL] + IDENTIFIER_COLS)
y = df[TARGET_COL]

print(f"Features shape: {X.shape}")
print(f"Target distribution:")
print(y.value_counts())

Features shape: (1885, 22)
Target distribution:
Scope_3_coverage
Yes              760
Not Specified    412
Partial          366
No               347
Name: count, dtype: int64


In [9]:
def create_stratified_splits(X, y, test_size=0.2, val_size=0.2, random_state=42):
    """Create stratified train/validation/test splits."""
    # First split: separate test set
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=random_state
    )
    
    # Second split: separate train/validation from remaining data
    val_size_adjusted = val_size / (1 - test_size)
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=val_size_adjusted, 
        stratify=y_temp, random_state=random_state
    )
    
    return X_train, X_val, X_test, y_train, y_val, y_test

# Apply split
X_train, X_val, X_test, y_train, y_val, y_test = create_stratified_splits(
    X, y, test_size=TEST_SIZE, val_size=VAL_SIZE, random_state=RANDOM_STATE
)

print("=" * 60)
print("📊 TRAIN-VALIDATION-TEST SPLIT")
print("=" * 60)
print(f"\nTraining set:   {X_train.shape[0]} rows ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validation set: {X_val.shape[0]} rows ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test set:       {X_test.shape[0]} rows ({X_test.shape[0]/len(X)*100:.1f}%)")

📊 TRAIN-VALIDATION-TEST SPLIT

Training set:   1131 rows (60.0%)
Validation set: 377 rows (20.0%)
Test set:       377 rows (20.0%)


## 4. Imputation (Using Training Statistics Only)

Calculate medians/modes from TRAINING data, apply to all sets.

In [10]:
# Identify numeric and categorical columns
numeric_cols = X_train.select_dtypes(include=['float64', 'int64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()

print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")

Numeric columns (4): ['End_target_year', 'Interim_target_year', 'Company_annual_revenue', 'Employees']
Categorical columns (18): ['Country', 'Geographic_region', 'Private_company', 'End_target', 'Status_of_end_target', 'Interim_target', 'GHGs_covered', 'Scope_1_coverage', 'Scope_2_coverage', 'Published_plan', 'Reporting_mechanism', 'Accountability_delivery', 'Carbon_credits', 'Separate_removal_target', 'Planning_removals', 'Historical_emissions', 'Race_to_zero_member', 'Industry']


In [11]:
# Calculate imputation values from TRAINING data only
GROUP_COL = 'Industry'  # Group by industry for more meaningful imputation

def calculate_imputation_stats(train_df, group_col, numeric_cols, categorical_cols):
    """
    Calculate imputation statistics from training data only.
    
    Returns:
        dict: {col: {group: value}} for group-based imputation
              {col: global_value} for global fallback
    """
    stats = {'group': {}, 'global': {}}
    
    # Numeric columns: group median + global median fallback
    for col in numeric_cols:
        if col == group_col:
            continue
        # Group medians
        group_medians = train_df.groupby(group_col)[col].median().to_dict()
        stats['group'][col] = group_medians
        # Global median fallback
        stats['global'][col] = train_df[col].median()
    
    # Categorical columns: group mode + global mode fallback
    for col in categorical_cols:
        if col == group_col:
            continue
        # Group modes
        group_modes = train_df.groupby(group_col)[col].agg(
            lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else np.nan
        ).to_dict()
        stats['group'][col] = group_modes
        # Global mode fallback
        mode_result = train_df[col].mode()
        stats['global'][col] = mode_result.iloc[0] if len(mode_result) > 0 else np.nan
    
    return stats

# Calculate stats from training data
imputation_stats = calculate_imputation_stats(X_train, GROUP_COL, numeric_cols, categorical_cols)
print("✅ Imputation statistics calculated from training data")
print(f"   Group column: {GROUP_COL}")
print(f"   Numeric columns with group medians: {len([c for c in numeric_cols if c != GROUP_COL])}")
print(f"   Categorical columns with group modes: {len([c for c in categorical_cols if c != GROUP_COL])}")

✅ Imputation statistics calculated from training data
   Group column: Industry
   Numeric columns with group medians: 4
   Categorical columns with group modes: 17


In [12]:
def apply_imputation(df, group_col, stats):
    """
    Apply imputation using pre-calculated statistics.
    
    Args:
        df: DataFrame to impute
        group_col: Column to group by
        stats: Dictionary with 'group' and 'global' imputation values
    
    Returns:
        DataFrame with imputed values
    """
    df_imputed = df.copy()
    
    for col in stats['group'].keys():
        if col not in df_imputed.columns:
            continue
            
        # Get group values for this column
        group_values = stats['group'][col]
        global_value = stats['global'][col]
        
        # Apply group-based imputation
        for idx in df_imputed[df_imputed[col].isna()].index:
            group = df_imputed.loc[idx, group_col]
            if pd.notna(group) and group in group_values and pd.notna(group_values[group]):
                df_imputed.loc[idx, col] = group_values[group]
            else:
                df_imputed.loc[idx, col] = global_value
    
    return df_imputed

# Apply imputation to all sets
print("\n🔄 Applying imputation...")
print(f"   Training set missing before: {X_train.isnull().sum().sum()}")
X_train_imputed = apply_imputation(X_train, GROUP_COL, imputation_stats)
print(f"   Training set missing after:  {X_train_imputed.isnull().sum().sum()}")

print(f"\n   Validation set missing before: {X_val.isnull().sum().sum()}")
X_val_imputed = apply_imputation(X_val, GROUP_COL, imputation_stats)
print(f"   Validation set missing after:  {X_val_imputed.isnull().sum().sum()}")

print(f"\n   Test set missing before: {X_test.isnull().sum().sum()}")
X_test_imputed = apply_imputation(X_test, GROUP_COL, imputation_stats)
print(f"   Test set missing after:  {X_test_imputed.isnull().sum().sum()}")

print("\n✅ Imputation complete - using TRAINING statistics only")


🔄 Applying imputation...
   Training set missing before: 1043
   Training set missing after:  0

   Validation set missing before: 365
   Validation set missing after:  0

   Test set missing before: 352
   Test set missing after:  0

✅ Imputation complete - using TRAINING statistics only


In [13]:
# Investigate remaining missing values
print("Remaining missing values by column (training set):")
remaining = X_train_imputed.isnull().sum()
print(remaining[remaining > 0])

print("\n--- Root cause: Industry column (grouping key) ---")
print(f"Industry missing in X_train: {X_train['Industry'].isnull().sum()}")

Remaining missing values by column (training set):
Series([], dtype: int64)

--- Root cause: Industry column (grouping key) ---
Industry missing in X_train: 0


## 5. Feature Engineering (Log Transforms)

Apply log transforms to skewed numeric features.

In [14]:
# Log transform skewed features
LOG_TRANSFORM_COLS = ['Company_annual_revenue', 'Employees', 'GHG_emissions']

def apply_log_transforms(df, cols):
    """Apply log1p transform to specified columns."""
    df_transformed = df.copy()
    for col in cols:
        if col in df_transformed.columns:
            # Create log-transformed column
            log_col = f'log_{col.lower().replace("company_annual_", "")}'
            df_transformed[log_col] = np.log1p(df_transformed[col].clip(lower=0))
    return df_transformed

# Apply to all sets
X_train_final = apply_log_transforms(X_train_imputed, LOG_TRANSFORM_COLS)
X_val_final = apply_log_transforms(X_val_imputed, LOG_TRANSFORM_COLS)
X_test_final = apply_log_transforms(X_test_imputed, LOG_TRANSFORM_COLS)

# Drop original columns (keep log versions)
cols_to_drop = [c for c in LOG_TRANSFORM_COLS if c in X_train_final.columns]
X_train_final = X_train_final.drop(columns=cols_to_drop)
X_val_final = X_val_final.drop(columns=cols_to_drop)
X_test_final = X_test_final.drop(columns=cols_to_drop)

print(f"Final feature set: {X_train_final.shape[1]} columns")
print(f"Columns: {X_train_final.columns.tolist()}")

Final feature set: 22 columns
Columns: ['Country', 'Geographic_region', 'Private_company', 'End_target', 'End_target_year', 'Status_of_end_target', 'Interim_target', 'Interim_target_year', 'GHGs_covered', 'Scope_1_coverage', 'Scope_2_coverage', 'Published_plan', 'Reporting_mechanism', 'Accountability_delivery', 'Carbon_credits', 'Separate_removal_target', 'Planning_removals', 'Historical_emissions', 'Race_to_zero_member', 'Industry', 'log_revenue', 'log_employees']


## 6. Save Preprocessed Data

In [15]:
# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Combine features and target
train_df = X_train_final.copy()
train_df[TARGET_COL] = y_train.values

val_df = X_val_final.copy()
val_df[TARGET_COL] = y_val.values

test_df = X_test_final.copy()
test_df[TARGET_COL] = y_test.values

# Save to CSV
train_path = os.path.join(OUTPUT_DIR, 'train.csv')
val_path = os.path.join(OUTPUT_DIR, 'val.csv')
test_path = os.path.join(OUTPUT_DIR, 'test.csv')

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

print("=" * 60)
print("✅ PREPROCESSING COMPLETE (No Data Leakage)")
print("=" * 60)
print(f"\nSaved files:")
print(f"   {train_path} ({train_df.shape[0]} rows, {train_df.shape[1]} columns)")
print(f"   {val_path} ({val_df.shape[0]} rows, {val_df.shape[1]} columns)")
print(f"   {test_path} ({test_df.shape[0]} rows, {test_df.shape[1]} columns)")

print(f"\n📋 Final columns: {train_df.columns.tolist()}")

✅ PREPROCESSING COMPLETE (No Data Leakage)

Saved files:
   ../data/interim/train.csv (1131 rows, 23 columns)
   ../data/interim/val.csv (377 rows, 23 columns)
   ../data/interim/test.csv (377 rows, 23 columns)

📋 Final columns: ['Country', 'Geographic_region', 'Private_company', 'End_target', 'End_target_year', 'Status_of_end_target', 'Interim_target', 'Interim_target_year', 'GHGs_covered', 'Scope_1_coverage', 'Scope_2_coverage', 'Published_plan', 'Reporting_mechanism', 'Accountability_delivery', 'Carbon_credits', 'Separate_removal_target', 'Planning_removals', 'Historical_emissions', 'Race_to_zero_member', 'Industry', 'log_revenue', 'log_employees', 'Scope_3_coverage']


## Summary

This preprocessing pipeline correctly handles data leakage by:

1. ✅ **Splitting BEFORE imputation** - Train/val/test split happens on raw data
2. ✅ **Training-only statistics** - Medians/modes calculated only from training set
3. ✅ **Consistent application** - Same imputation values applied to val/test sets
4. ✅ **No information leakage** - Test set never influences preprocessing decisions

